In [1]:
import os
os.chdir('..')
from gpu_management import set_gpus
import sys

os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform' 
# os.environ['CUDA_VISIBLE_DEVICES'] = '1' 
set_gpus(1, forcing=True)
print(f"XLA_PYTHON_CLIENT_ALLOCATOR set to: {os.environ.get('XLA_PYTHON_CLIENT_ALLOCATOR')}")
print(f"CUDA_VISIBLE_DEVICES set to: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

XLA_PYTHON_CLIENT_ALLOCATOR set to: platform
CUDA_VISIBLE_DEVICES set to: 1


In [2]:
from data import DL17Lands
import polars as pl

dataloader = DL17Lands('FDN', verbose=True)

Loading FDN
-----------
Loading card data: 0.152087s
Making extension dataframe: 0.698368s
##################################################


In [4]:
import torch
from sentence_transformers import SentenceTransformer

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/embeddinggemma-300M"
model = SentenceTransformer(model_id).to(device=device)

print(f"Device: {model.device}")
print(model)
print("Total number of parameters in the model:", sum([p.numel() for _, p in model.named_parameters()]))

Device: cuda:0
SentenceTransformer(
  (0): Transformer({'max_seq_length': 2048, 'do_lower_case': False, 'architecture': 'Gemma3TextModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 3072, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'})
  (3): Dense({'in_features': 3072, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'})
  (4): Normalize()
)
Total number of parameters in the model: 307581696


In [6]:
card_encodings = model.encode(dataloader.cards['oracle'])

In [8]:
model.similarity(card_encodings[0], card_encodings[2]), dataloader.cards['name'][0], dataloader.cards['name'][2]

(tensor([[0.4861]]), 'Sire of Seven Deaths', 'Armasaur Guide')

In [10]:
#card_encodings_prompted = model.encode(dataloader.cards['oracle'], prompt="task: Magic the Gathering card evaluation | card: ")
card_encodings_prompted = model.encode(dataloader.cards['oracle'], prompt="task: Magic the Gathering card selection in a draft | card: ")

In [13]:
from card_utils import make_oracle
#split_cards = dataloader.all_cards.filter(pl.col('name').str.contains('//'))['name'].str.split(by=' // ').explode()
#all_cards = dataloader.all_cards.filter(~pl.col('name').is_in(split_cards.implode())).unique(subset='name')

df_cards = dataloader.all_cards
# Split cards are listed 3 times (once fully, then once for each half).
# so we drop the half cards, then rename the full to keep only the
# first half
split_cards = (
    df_cards.filter(pl.col('name').str.contains('//'))['name']
    .str.split(by=' // ')
).explode()
df_cards = df_cards.filter(
    ~pl.col('name').is_in(split_cards.implode())
)
df_cards = df_cards.with_columns(
    name=pl.col('name').str.split(by=' // ').list.get(0)
)
df_cards = df_cards.unique(subset='name')

oracle = make_oracle(df_cards)
df_cards = df_cards.with_columns(oracle=oracle['oracle'], full_name=oracle['name'])
df_cards = df_cards.filter(~pl.col('oracle').is_null())
all_cards = df_cards

In [14]:
%%time
all_cards_encodings = model.encode(all_cards['oracle'])
all_cards_encodings

CPU times: user 1min 59s, sys: 19.3 s, total: 2min 19s
Wall time: 51.6 s


array([[-0.07377799, -0.0282974 , -0.01578678, ..., -0.04332677,
         0.00483069, -0.0103206 ],
       [-0.07186811,  0.01036564, -0.03428526, ..., -0.09684604,
         0.03789444, -0.04156143],
       [-0.06555114, -0.01909189, -0.02412035, ..., -0.00320677,
         0.01699104, -0.02693046],
       ...,
       [-0.0750476 , -0.0125245 , -0.01682081, ..., -0.02712863,
         0.02660303, -0.00535699],
       [-0.07830209, -0.02078592, -0.05052879, ..., -0.05028763,
         0.0093497 , -0.03834897],
       [-0.04744077, -0.07003206, -0.05155687, ..., -0.0578826 ,
        -0.03454551, -0.02954187]], shape=(13802, 768), dtype=float32)

In [15]:
sim_matrix = model.similarity(card_encodings, card_encodings)
prompted_sim_matrix = model.similarity(card_encodings_prompted, card_encodings_prompted)

In [16]:
%%time
all_sim_matrix = model.similarity(all_cards_encodings, all_cards_encodings)

CPU times: user 10.5 s, sys: 2.22 s, total: 12.8 s
Wall time: 268 ms


In [18]:
def summary(card, n=5):
    print(dataloader.cards['name'][card])
    print('===== Default =====')
    scores, closest = torch.topk(sim_matrix[card], k=n)
    print([dataloader.cards['name'][i.item()] for i in closest])
    print(scores)
    scores, closest = torch.topk(-sim_matrix[card], k=n)
    print([dataloader.cards['name'][i.item()] for i in closest.flip(0)])
    print(-scores.flip(0))
    print('===== Prompted =====')
    scores, closest = torch.topk(prompted_sim_matrix[card], k=n)
    print([dataloader.cards['name'][i.item()] for i in closest])
    print(scores)
    scores, closest = torch.topk(-prompted_sim_matrix[card], k=n)
    print([dataloader.cards['name'][i.item()] for i in closest.flip(0)])
    print(-scores.flip(0))
    print('===== All Cards =====')
    all_id = all_cards.with_row_index().filter(pl.col('name') == dataloader.cards['name'][card])['index'][0]
    scores, closest = torch.topk(all_sim_matrix[all_id], k=n)
    print([all_cards['name'][i.item()] for i in closest])
    print(scores)
    scores, closest = torch.topk(-all_sim_matrix[all_id], k=n)
    print([all_cards['name'][i.item()] for i in closest.flip(0)])
    print(-scores.flip(0))

summary(1)

Arahbo, the First Fang
===== Default =====
['Arahbo, the First Fang', 'Cat Collector', "Ajani's Pridemate", 'Prideful Parent', 'Ajani, Caller of the Pride']
tensor([1.0000, 0.6619, 0.6403, 0.6291, 0.6091])
['Evolving Wilds', 'Omniscience', 'Grim Tutor', 'Burst Lightning', 'Incinerating Blast']
tensor([0.3914, 0.3823, 0.3822, 0.3796, 0.3796])
===== Prompted =====
['Arahbo, the First Fang', 'Cat Collector', 'Helpful Hunter', "Ajani's Pridemate", 'Ajani, Caller of the Pride']
tensor([1.0000, 0.7523, 0.7334, 0.7314, 0.7289])
['Incinerating Blast', 'Swiftwater Cliffs', 'Time Stop', 'Dismal Backwater', 'Grim Tutor']
tensor([0.5562, 0.5552, 0.5513, 0.5484, 0.5449])
===== All Cards =====
['Arahbo, the First Fang', 'Kemba, Kha Enduring', 'Cat Collector', 'Prowling Caracal', 'Envoy of Okinec Ahau']
tensor([1.0000, 0.6798, 0.6619, 0.6597, 0.6561])
["Roamer's Routine", 'You Find a Cursed Idol', 'Invasive Surgery', "Mizzix's Mastery", 'Damn']
tensor([0.3410, 0.3393, 0.3365, 0.3331, 0.3193])


In [19]:
from card_utils import get_card_image

In [20]:
import html
from IPython.display import display, HTML, Markdown

def create_card_row_html(names, scores):
    html_cards = []
    for name, score in zip(names, scores):
        url = get_card_image(name)
        score_label = f'{score:.4f}'
        html_cards.append(f"""
        <div style="display: inline-block; margin: 0 5px; text-align: center; width: 108px">
            <img src="{url}" style="height: 150px; border-radius: 5px; box-shadow: 2px 2px 5px #888888;">
            <p style="margin: 5px 0 0 0; font-size: 12px; font-weight: bold;">{name}</p>
            <p style="margin: 0; font-size: 11px; color: #555;">Score: {score_label}</p>
        </div>
        """)
    return f'<div style="display: flex; flex-direction: row; justify-content: flex-start; flex-wrap: nowrap;">{"".join(html_cards)}</div>'


def create_side_by_side_block_html(title, left, right):
    return f"""
    <div style="border: 1px solid #ddd; padding: 10px; border-radius: 5px;">
        <h3 style="text-align: center; color: #333;">{title}</h3>
        <div style="display: flex; flex-direction: row; align-items: flex-start; overflow-x: auto; padding: 10px 0;">
            <div style="flex-shrink: 0; margin-right: 20px;">
                <h4 style="text-align: center; margin-top: 0;">Closest Cards</h4>
                {create_card_row_html(left['names'], left['scores'])}
            </div>
            <div style="width: 1px; height: 250px; background-color: #bbb; margin: 0 20px; flex-shrink: 0;"></div>
            <div style="flex-shrink: 0; margin-left: 20px;">
                <h4 style="text-align: center; margin-top: 0;">Furthest Cards</h4>
                {create_card_row_html(right['names'], right['scores'])}
            </div>
        </div>
    </div>
    """

def card_summary(card_id, k=5):
    card_name = dataloader.cards['name'][card_id]
    scoresL, cardsL = torch.topk(sim_matrix[card_id], k=k+1)
    scoresL, cardsL = scoresL[1:], cardsL[1:]
    scoresR, cardsR = torch.topk(sim_matrix[card_id], k=k, largest=False)
    default_html = create_side_by_side_block_html(
        "Default Similarity Results",
        {'names': [dataloader.cards['name'][i.item()] for i in cardsL], 'scores': scoresL},
        {'names': [dataloader.cards['name'][i.item()] for i in cardsR.flip(0)], 'scores': scoresR.flip(0)}
    )
    scoresL, cardsL = torch.topk(prompted_sim_matrix[card_id], k=k+1)
    scoresL, cardsL = scoresL[1:], cardsL[1:]
    scoresR, cardsR = torch.topk(prompted_sim_matrix[card_id], k=k, largest=False)
    prompted_html = create_side_by_side_block_html(
        "Prompted Similarity Results",
        {'names': [dataloader.cards['name'][i.item()] for i in cardsL], 'scores': scoresL},
        {'names': [dataloader.cards['name'][i.item()] for i in cardsR.flip(0)], 'scores': scoresR.flip(0)}
    )
    all_id = all_cards.with_row_index().filter(pl.col('name') == dataloader.cards['name'][card_id])['index'][0]
    scoresL, cardsL = torch.topk(all_sim_matrix[all_id], k=k+1)
    scoresL, cardsL = scoresL[1:], cardsL[1:]
    scoresR, cardsR = torch.topk(all_sim_matrix[all_id], k=k, largest=False)
    allcards_html = create_side_by_side_block_html(
        "All Cards Similarity Results",
        {'names': [all_cards['name'][i.item()] for i in cardsL], 'scores': scoresL},
        {'names': [all_cards['name'][i.item()] for i in cardsR.flip(0)], 'scores': scoresR.flip(0)}
    )
    display(Markdown("# Card Similarity Analysis"))
    display(HTML(f"""
    <div style="display: flex; flex-direction: row; align-items: flex-start; width: 1550px; height: 1000px">
        <div style="flex-shrink: 0; margin-right: 40px; text-align: center; width: 220px;">
            <h2>Initial Card</h2>
            <h3 style="color: #6A0DAD;">{card_name}</h3>
            <img src="{get_card_image(card_name)}" style="max-width: 100%; height: auto; border-radius: 10px; box-shadow: 0 6px 12px 0 rgba(0, 0, 0, 0.4);">
            <p>{html.escape(dataloader.cards['oracle'][card_id])}</p>
        </div>
        <div style="flex-grow: 1;">
            {default_html}
            {prompted_html}
            {allcards_html}
        </div>
    </div>
    """))
card_summary(0)

# Card Similarity Analysis

In [21]:
card_summary(10)

# Card Similarity Analysis

In [22]:
card_summary(dataloader.cards.with_row_index().filter(pl.col('name') == "Hidetsugu's Second Rite")['index'][0])

# Card Similarity Analysis

In [23]:
card_summary(dataloader.cards.with_row_index().filter(pl.col('name') == "Abrade")['index'][0])

# Card Similarity Analysis